# Week 17, Day 5: The bias-variance trade-off, cross-validation and model selection

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w017/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Measure the trade-off

For degrees 0 to 8, fit polynomials to 500 datasets of 20 points (noise sd 0.4). On a fixed grid of test inputs inside the data range, compute bias², variance and the expected test error (including fresh noise), averaged over the grid, plus the average training error. Store arrays `bias2`, `var`, `pred_err` and `train_err`, indexed by degree.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(2019)
def truth(x): return np.sin(2.5 * x) + 0.3 * x
xg = np.linspace(-1.2, 1.2, 50)
degrees = range(9)
bias2, var, pred_err, train_err = [], [], [], []
for d in degrees:
    preds, tr = [], []
    for _ in range(500):
        x = rng.uniform(-1.5, 1.5, 20)
        y = truth(x) + 0.4 * rng.normal(size=20)
        c = np.polyfit(x / 1.5, y, d)
        preds.append(np.polyval(c, xg / 1.5))
        tr.append(np.mean((np.polyval(c, x / 1.5) - y) ** 2))
    preds = np.array(preds)
    mean_pred = preds.mean(axis=0)
    bias2.append(...)
    var.append(...)
    pred_err.append(np.mean((preds - truth(xg)) ** 2) + 0.16)    # expected error on a fresh noisy point
    train_err.append(np.mean(tr))
bias2, var, pred_err, train_err = map(np.array, (bias2, var, pred_err, train_err))
print("degree   bias^2  variance  test   train")
for d in degrees: print(f"{d:5d}  {bias2[d]:7.3f}  {var[d]:8.3f}  {pred_err[d]:5.3f}  {train_err[d]:5.3f}")

In [ ]:
import numpy as np
def test_decomposition():
    "for every degree, bias^2 + variance + noise matches the measured test error to within 10%"
    assert np.all(np.abs(bias2 + var + 0.16 - pred_err) / pred_err < 0.1)

def test_shapes():
    "bias^2 falls and variance rises from degree 1 to degree 7, and training error falls throughout"
    assert bias2[7] < bias2[1] and var[7] > var[1] and np.all(np.diff(train_err) <= 1e-9)

def test_u():
    "test error is lowest at an intermediate degree (between 2 and 7)"
    assert 2 <= int(np.argmin(pred_err)) <= 7

test_decomposition()
test_shapes()
test_u()
print("All checks passed")

### Exercise: Choose the degree by cross-validation

Write `cv_error(x, y, degree, k=5)` that returns the mean squared error on held-out folds. Using one dataset of 40 points, compute the cross-validated error for degrees 0 to 8 (`cv_errs`), pick `best_cv`, and compare its true test error (from the previous cell) with the best possible.

In [ ]:
rng = np.random.default_rng(7)
x40 = rng.uniform(-1.5, 1.5, 40)
y40 = truth(x40) + 0.4 * rng.normal(size=40)

def cv_error(x, y, degree, k=5):
    folds = np.array_split(rng.permutation(len(x)), k)
    errs = []
    for f in folds:
        train = np.setdiff1d(np.arange(len(x)), f)
        c = ...
        errs.append(...)
    return np.mean(errs)

cv_errs = np.array([cv_error(x40, y40, d) for d in range(9)])
best_cv = int(np.argmin(cv_errs))
print("cross-validated error by degree:", cv_errs.round(3))
print(f"chosen degree {best_cv}; its true test error {pred_err[best_cv]:.3f}; best possible {pred_err.min():.3f} at degree {int(np.argmin(pred_err))}")

In [ ]:
import numpy as np
def test_cv():
    "cv_error returns one mean held-out error per degree"
    assert len(cv_errs) == 9 and np.all(np.isfinite(cv_errs))

def test_choice():
    "the degree chosen by cross-validation has a true test error within 25% of the best possible"
    assert pred_err[best_cv] < 1.25 * pred_err.min()

test_cv()
test_choice()
print("All checks passed")

### Exercise: AIC and BIC

For degrees 0 to 8 fitted to the 40 points, compute the residual sum of squares, then AIC and BIC with p = degree + 2. Store arrays `aic` and `bic` and compare their choices with cross-validation.

In [ ]:
aic, bic = [], []
for d in range(9):
    c = np.polyfit(x40 / 1.5, y40, d)
    rss = np.sum((np.polyval(c, x40 / 1.5) - y40) ** 2)
    p, n = d + 2, len(x40)
    aic.append(...)
    bic.append(...)
aic, bic = np.array(aic), np.array(bic)
print(f"chosen degree: AIC {int(np.argmin(aic))}, BIC {int(np.argmin(bic))}, cross-validation {best_cv}")

In [ ]:
import numpy as np
def test_formula():
    "aic and bic use n log(RSS / n) with p = degree + 2 parameters (coefficients plus the noise variance)"
    c = np.polyfit(x40 / 1.5, y40, 3); rss = np.sum((np.polyval(c, x40 / 1.5) - y40) ** 2)
    assert abs(aic[3] - (2 * 5 + 40 * np.log(rss / 40))) < 1e-9 and abs(bic[3] - (5 * np.log(40) + 40 * np.log(rss / 40))) < 1e-9

def test_bic_simpler():
    "BIC never chooses a more complex model than AIC here"
    assert int(np.argmin(bic)) <= int(np.argmin(aic))

test_formula()
test_bic_simpler()
print("All checks passed")